# Question 1.

In [2]:
pip install ucimlrepo

In [3]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
spambase = fetch_ucirepo(id=94)

# data (as pandas dataframes)
X = spambase.data.features
y = spambase.data.targets

# metadata
print(spambase.metadata)

# variable information
print(spambase.variables)


{'uci_id': 94, 'name': 'Spambase', 'repository_url': 'https://archive.ics.uci.edu/dataset/94/spambase', 'data_url': 'https://archive.ics.uci.edu/static/public/94/data.csv', 'abstract': 'Classifying Email as Spam or Non-Spam', 'area': 'Computer Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 4601, 'num_features': 57, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1999, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C53G6X', 'creators': ['Mark Hopkins', 'Erik Reeber', 'George Forman', 'Jaap Suermondt'], 'intro_paper': None, 'additional_info': {'summary': 'The "spam" concept is diverse: advertisements for products/web sites, make money fast schemes, chain letters, pornography...\n\nThe classification task for this dataset is to determine whether a given email is spam or not.\n\t\nOur collecti

In [4]:
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score,precision_score,
                             recall_score, f1_score)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 50,
                                                    stratify = y)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter = 5000, solver = "lbfgs"))])

model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
error = 1 - accuracy
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Confusion Matrix:")
print(cm)
print(f"Accuracy : {accuracy:.4f}")
print(f"Error    : {error:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

coef = model.named_steps["logreg"].coef_[0]
coef_df = pd.DataFrame({"feature": X.columns, "coefficient": coef, "abs_coefficient": np.abs(coef)})

print("\nTop 15 features by absolute coeffecient:")
print(coef_df.sort_values("abs_coefficient", ascending = False).head(15))
print("\nMost positive coefficients:")
print(coef_df.sort_values("coefficient" , ascending = False).head(10))
print("\nMost negative coefficients:")
print(coef_df.sort_values("coefficient", ascending=True).head(10))


Confusion Matrix:
[[656  41]
 [ 48 406]]
Accuracy : 0.9227
Error    : 0.0773
Precision: 0.9083
Recall   : 0.8943
F1 Score : 0.9012

Top 15 features by absolute coeffecient:
              feature  coefficient  abs_coefficient
26   word_freq_george    -4.485202         4.485202
24       word_freq_hp    -2.371194         2.371194
25      word_freq_hpl    -1.676559         1.676559
40       word_freq_cs    -1.642827         1.642827
41  word_freq_meeting    -1.434536         1.434536
45      word_freq_edu    -1.213922         1.213922
52        char_freq_$     1.159642         1.159642
28      word_freq_lab    -1.043206         1.043206
15     word_freq_free     1.037989         1.037989
43  word_freq_project    -1.021475         1.021475
34       word_freq_85    -1.003415         1.003415
22      word_freq_000     0.967152         0.967152
3        word_freq_3d     0.910981         0.910981
44       word_freq_re    -0.899286         0.899286
6    word_freq_remove     0.869947         0.86

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


**Which features contribute mostly to the prediction? Which ones are positively
correlated and which ones are negatively correlated with the SPAM class?** The most influencial features overall were word_freq_george (-4.4853), word_freq_hp (-2.3712), word_freq_hpl (-1.6766), and word_freq_cs (-1.6428). The strongest features with positive coefficients were char_freq_$ (1.1596), word_freq_free (1.0380), and word_freq_000 (0.9672), These were the features that were the most positively correlated with spam. The strongest negative features were word_freq_george (-4.4852), word_freq_hp (-2.3712), and word_freq_hpl (-1.6766). All the strong features were more typical for a workplace or academic environment, so they indicate the prediction toward ham.


In [9]:
thresholds = [0.25, 0.5, 0.75, 0.9]
rows = []

for t in thresholds:
  pred_t = (y_prob >= t).astype(int)
  rows.append({
      "threshold": t,
      "accuracy" : accuracy_score(y_test, pred_t),
      "precision" : precision_score(y_test, pred_t, zero_division = 0),
      "recall" : recall_score(y_test, pred_t, zero_division = 0)
  })
  threshold_df = pd.DataFrame(rows)
  print("\nThreshold results: ")
  print(threshold_df)


Threshold results: 
   threshold  accuracy  precision    recall
0       0.25  0.891399   0.807477  0.951542

Threshold results: 
   threshold  accuracy  precision    recall
0       0.25  0.891399   0.807477  0.951542
1       0.50  0.922676   0.908277  0.894273

Threshold results: 
   threshold  accuracy  precision    recall
0       0.25  0.891399   0.807477  0.951542
1       0.50  0.922676   0.908277  0.894273
2       0.75  0.886186   0.940054  0.759912

Threshold results: 
   threshold  accuracy  precision    recall
0       0.25  0.891399   0.807477  0.951542
1       0.50  0.922676   0.908277  0.894273
2       0.75  0.886186   0.940054  0.759912
3       0.90  0.836664   0.952381  0.616740


**Vary the decision threshold $T \in \{0.25,0.5,0.75,0.9\}$ and report for each value the model accuracy, precision, and recall. Comment on
how these metrics vary with the choice of threshold.** As the decision threshold increases, the classifier becomes more conservative about predicting spam. At a low threshold like 0.25, the model labels more emails as spam, giving the highest recall of 0.9515, but the precision is lower (0.8075) meaning it catches the most spam emails, but it also produces more false positives. At the threshold of 0.5, the model gives the best overall balance with an accuracy of 0.9227 and strong precision and recall. At higher thresholds, the model become stricter, so precision increases (0.9401 and 0.9524) but recall decreases (0.7599 and 0.6167) because the model misses more actual spam emails.